# Walkthrough

Reads the artefacts the pipeline writes and reproduces the report's headline numbers. **Nothing here trains a model**; every cell reads files that already
exist. Before running it:

```bash
pip install -e .
python -m ecg_classification.mitdb --build-cache
python -m ecg_classification.train --protocol intra --seed 42
python -m ecg_classification.train --protocol inter --seed 42
```

The full write-up is in [`docs/report.md`](../docs/report.md).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from ecg_classification.constants import CLASS_SYMBOLS, DS1, DS2
from analysis.bootstrap import (
    cluster_bootstrap,
    confusion,
    intraclass_correlation,
    load_predictions,
    macro_f1,
    naive_bootstrap,
    per_class_f1,
    percentile_interval,
)
from analysis.records import effective_records, per_record_table

# The notebook lives in notebooks/; everything it reads is relative to the root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, OUTPUTS = ROOT / "data", ROOT / "outputs"

REPS = 1000
rng = np.random.default_rng(0)

## 1. Classes are nested inside patients

The effective number of contributing recordings is the inverse Simpson index, $N_{\text{eff}} = 1 / \sum_r p_r^2$. It equals the number of recordings when a
class is spread evenly and falls to 1 when one recording supplies all of it. MIT-BIH holds one recording per subject, so this is a count of patients.

In [ ]:
counts = pd.read_csv(DATA / "narrow" / "class_record_counts.csv", dtype={"record_id": str})

rows = []
for half, records in (("DS1 (train)", DS1), ("DS2 (test)", DS2)):
    part = counts[counts.record_id.isin(records)]
    for symbol in CLASS_SYMBOLS:
        column = part[symbol].to_numpy(dtype=float)
        rows.append({
            "half": half,
            "class": symbol,
            "beats": int(column.sum()),
            "N_eff": round(effective_records(column), 2),
            "largest share": f"{column.max() / column.sum():.0%}",
        })

pd.DataFrame(rows).set_index(["half", "class"])

## 2. Separating patients costs about half the score

The same network and preprocessing under both protocols. Intervals come from a cluster bootstrap that resamples **recordings**, not beats.

In [ ]:
runs = {
    "intra": "intra-none-lossweight0-seed42",
    "inter": "inter-none-lossweight0-seed42",
}
frames = {name: load_predictions(OUTPUTS / run) for name, run in runs.items()}

rows = []
for name, frame in frames.items():
    y_true, y_pred = frame.y_true.to_numpy(), frame.y_pred.to_numpy()
    matrix = confusion(y_true, y_pred)
    replicates, _, _ = cluster_bootstrap(y_true, y_pred, frame.record_id.to_numpy(), REPS, rng)
    low, high = percentile_interval(replicates)
    rows.append({
        "protocol": name,
        **{symbol: round(value, 3) for symbol, value in zip(CLASS_SYMBOLS, per_class_f1(matrix))},
        "macro (N/S/V)": round(macro_f1(matrix), 4),
        "95% CI": f"[{low:.3f}, {high:.3f}]",
    })

pd.DataFrame(rows).set_index("protocol")

## 3. The recording is the sampling unit

Correctness is clustered within recordings. The design effect $D_{\text{eff}} = 1 + (\bar m - 1)\rho$ converts that clustering into an
effective sample size, and predicts how much narrower a beat-level interval will be than a recording-level one.

The same statistic reads very differently under the two protocols: a beat-level split hides the structure that inflates it.

In [ ]:
for name, frame in frames.items():
    correct = (frame.y_true == frame.y_pred).to_numpy(dtype=float)
    stats = intraclass_correlation(correct, frame.record_id.to_numpy())
    print(f"{name:>5}   ICC {stats['icc']:.3f}   design effect {stats['design_effect']:7.1f}"
          f"   effective n {stats['n_effective']:6.0f} of {len(frame):,} beats")

In [ ]:
frame = frames["inter"]
y_true, y_pred = frame.y_true.to_numpy(), frame.y_pred.to_numpy()

cluster, _, _ = cluster_bootstrap(y_true, y_pred, frame.record_id.to_numpy(), REPS, rng)
beat, _ = naive_bootstrap(y_true, y_pred, REPS, rng)

widths = {}
for name, replicates in (("beats resampled", beat), ("recordings resampled", cluster)):
    low, high = percentile_interval(replicates)
    widths[name] = high - low
    print(f"{name:>21}   [{low:.4f}, {high:.4f}]   width {high - low:.4f}")

print(f"\nThe recording-level interval is {widths['recordings resampled'] / widths['beats resampled']:.0f} times wider.")

## 4. The penalty is concentrated in a few patients

Accuracy per test recording under the inter-patient protocol, worst first.

In [ ]:
table = per_record_table(frames["inter"])
errors = (1 - table.accuracy) * table.n
share = errors.nlargest(3).sum() / errors.sum()

print(f"the three weakest recordings carry {share:.0%} of all errors\n")
table[["record_id", "n", "accuracy", "f1_N", "f1_S", "f1_V", "n_S"]].head(6).round(3)

## Further

Everything in the report is generated from these same artefacts:

```bash
python -m analysis.bootstrap   outputs/inter-none-lossweight0-seed42
python -m analysis.records     outputs/inter-none-lossweight0-seed42 --intra outputs/intra-none-lossweight0-seed42
python -m analysis.calibration outputs/inter-none-lossweight0-seed42
python -m analysis.baselines   --protocol intra --protocol inter
python -m analysis.figures
```

Representation arms are selected with `ECG_REPRESENTATION`; see the README.